In [ ]:
%load_ext autoreload
%autoreload 2

import pylupnt as pnt
import numpy as np
from src.failure_model import NavSatFailureModel
from src.constellation_design import setup_problem_config

from src.constellation_design import ConstellationOptimization
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.core.mixed import MixedVariableGA
from pymoo.algorithms.moo.nsga2 import RankAndCrowdingSurvival
import time

## Setup

In [ ]:
# config for test
objs = ["dop", "nsat"]
et0 = pnt.convert_time(pnt.gregorian_to_time(2025, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)

sim_days = 1
steps_per_hr = 6
tspan = np.linspace(0, sim_days * pnt.SECS_DAY, sim_days * 24 * steps_per_hr + 1)

# satellite fail model
failmodel = NavSatFailureModel(
    tau1=0.25,  # years (≈3 months)
    S_tau1=0.95,  # survival at tau1
    beta0=0.6,  # infant mortality shape
    lam1=0.008,  # /year, nominal constant hazard
    tau2=5.0,  # start of wear-out (years)
    targets_years=(6.0, 8.0),
    targets_survival=(0.8, 0.5),  # survival at 6 and 8 years
)  # defaults described above

# config
config = setup_problem_config(
    objs,
    n_walker=2,
    n_phase=2,
    sat_range_phases=[[3, 8], [15, 30]],
    n_users=200,
    elev_mask_deg=5.0,
    cn0_thresh_user=30.0,
    dop_type_phase=["hdop", "pdop"],
    use_wdop=False,
    dop_target=[6.0, 2.0],
    compute_link_budget=True,
    lat_masks=[[-90, -75], [-90, 90]],
    launch_years=[0, 5],
    eval_years=[1, 6],
    fail_model=failmodel,
    max_fail_sat=1,
    epoch=et0,
    tspan=tspan,
)

## Run optimization

In [ ]:
problem = ConstellationOptimization(config)

algorithm = MixedVariableGA(pop_size=100, survival=RankAndCrowdingSurvival())

termination = get_termination("n_gen", 5)

start_time = time.time()
res = minimize(problem, algorithm, termination, seed=1, save_history=True, verbose=True)

end_time = time.time()
print("Optimization completed in {:.2f} seconds".format(end_time - start_time))

## Postprocess

In [ ]:
import matplotlib.pyplot as plt
from src.constellation_design import N_SAT_MAX

X = res.X
F = res.F

fig, axes = plt.subplots(2, 2, figsize=(10, 6))

#
axes[0, 0].scatter(
    1 - F[:, 0], F[:, 1] * N_SAT_MAX, s=30, facecolors="none", edgecolors="blue"
)
axes[0, 0].set_xlabel("DOP under 6 ratio")
axes[0, 0].set_ylabel("NSAT")
axes[0, 0].set_title("Objective Space (Phase 1)")
axes[0, 0].grid(True)

axes[0, 1].scatter(
    1 - F[:, 2], F[:, 3] * N_SAT_MAX, s=30, facecolors="none", edgecolors="blue"
)
axes[0, 1].set_xlabel("DOP under 3 ratio")
axes[0, 1].set_ylabel("NSAT")
axes[0, 1].set_title("Objective Space (Phase 2)")
axes[0, 1].grid(True)

axes[1, 0].scatter(1 - F[:, 0], 1 - F[:, 2], s=30, facecolors="none", edgecolors="blue")
axes[1, 0].set_xlabel("Phase 1: DOP under 6 ratio")
axes[1, 0].set_ylabel("Phase 2: DOP under 3 ratio")
axes[1, 0].set_title("Objective Space (Phase 1 vs Phase 2)")
axes[1, 0].grid(True)

axes[1, 1].scatter(
    (1 - F[:, 0] + 1 - F[:, 2]) / 2,
    F[:, 1] * N_SAT_MAX,
    s=30,
    facecolors="none",
    edgecolors="blue",
)
axes[1, 1].set_xlabel("Phase 1: DOP under 6 ratio + Phase 2: DOP under 3 ratio")
axes[1, 1].set_ylabel("NSAT")
axes[1, 1].set_title("Objective Space (Combined DOP vs NSAT)")
axes[1, 1].grid(True)

fig.tight_layout()

In [ ]:
from pymoo.visualization.scatter import Scatter

plot = Scatter(tight_layout=True)
plot.add(F, s=10)
plot.add(F[10], s=30, color="red")
plot.show()

In [ ]:
from src.postprocess import plot_history_hv

plot_history_hv(res)

In [ ]:
import pickle

with open("data/test_opt_result.pkl", "wb") as f:
    pickle.dump(res, f)

In [ ]:
# load
with open("data/test_opt_result.pkl", "rb") as f:
    res_load = pickle.load(f)

from src.postprocess import plot_history_hv

plot_history_hv(res_load)